
# Task A XGBoost Classifier + Optuna

This notebook adds a **single-model `xgboost.XGBClassifier`** to the existing Task A workflow without changing preprocessing.

## What it reuses from the current Task A ensemble
- raw train / validation / test I/O from `credit_train.csv`, `credit_test.csv`, and `sample_submission.csv`
- the existing Task A preprocessing contract (`fit_task_a_preprocessor` / `transform_task_a`)
- the existing submission behavior that preserves `InterestRate` from `submissions/submission.csv` when available

## Assumptions about the current classifier interface
- Task A models accept raw pandas `DataFrame` inputs and integer `RiskTier` targets.
- Preprocessing is fit on training data only and then reused for validation / test, so leakage must stay out of the flow.
- Validation metrics should include accuracy, macro F1, confusion matrix, and a full classification report.
- The current task is **multiclass** (`RiskTier` in `[0, 1, 2, 3, 4]`), so `scale_pos_weight` is left disabled by default and is only meaningful if the task ever becomes binary.

## Dependencies
Install these in the notebook environment before running training or tuning:

```bash
pip install xgboost optuna joblib
```


In [ ]:

from __future__ import annotations

import importlib.util
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
MODULE_PATH = REPO_ROOT / 'task_a' / 'scripts' / 'taskA_xgb_optuna.py'

spec = importlib.util.spec_from_file_location('taskA_xgb_optuna', MODULE_PATH)
taska = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = taska
assert spec.loader is not None
spec.loader.exec_module(taska)

print('Loaded module:', MODULE_PATH)
print('Train CSV:', taska.TRAIN_PATH)
print('Test CSV :', taska.TEST_PATH)
print('Submission target:', taska.SUBMISSION_PATH)


In [ ]:

TRAIN_CONFIG = taska.TrainEvalConfig(
    artifact_tag='taskA_xgb_trainval',
    validation_size=0.2,
    random_state=42,
    model_threads=2,
    metric_name='accuracy',
    enable_early_stopping=True,
    early_stopping_rounds=40,
    internal_early_stopping_size=0.1,
    run_full_train_prediction=True,
    preserve_interest_rate=True,
    xgb_params={
        'n_estimators': 600,
        'max_depth': 6,
        'learning_rate': 0.05,
        'min_child_weight': 1.0,
        'subsample': 0.9,
        'colsample_bytree': 0.8,
        'gamma': 0.0,
        'reg_alpha': 0.0,
        'reg_lambda': 1.0,
    },
    use_scale_pos_weight=False,
)

print(json.dumps(taska.asdict(TRAIN_CONFIG), indent=2, default=taska._json_default))


In [ ]:

RUN_VALIDATION = False

if RUN_VALIDATION:
    train_results = taska.run_train_validation_experiment(TRAIN_CONFIG)
    print(json.dumps(train_results, indent=2, default=taska._json_default))
else:
    print('Validation run skipped. Set RUN_VALIDATION=True to train and save artifacts.')


In [ ]:

OPTUNA_CONFIG = taska.OptunaConfig(
    study_name='taskA_xgb_optuna',
    artifact_tag='taskA_xgb_optuna',
    n_trials=25,
    timeout=None,
    cv_folds=5,
    metric_name='accuracy',
    random_state=42,
    model_threads=2,
    enable_early_stopping=True,
    early_stopping_rounds=40,
    internal_early_stopping_size=0.1,
    include_scale_pos_weight=False,
    fixed_params={},
)

print(json.dumps(taska.asdict(OPTUNA_CONFIG), indent=2, default=taska._json_default))


In [ ]:

RUN_TUNING = False

if RUN_TUNING:
    tuning_results = taska.run_optuna_study(OPTUNA_CONFIG)
    print(json.dumps(tuning_results, indent=2, default=taska._json_default))
else:
    print('Optuna run skipped. Set RUN_TUNING=True to start Bayesian optimization.')



## Example terminal commands

Train / validate and then generate a fresh submission using the full training set:

```bash
python task_a/scripts/taskA_xgb_optuna.py train           --artifact-tag taskA_xgb_trainval           --validation-size 0.2           --metric-name accuracy           --early-stopping-rounds 40           --xgb-param n_estimators=600           --xgb-param max_depth=6           --xgb-param learning_rate=0.05
```

Run Optuna tuning on the training data only:

```bash
python task_a/scripts/taskA_xgb_optuna.py tune           --study-name taskA_xgb_optuna           --artifact-tag taskA_xgb_optuna           --n-trials 40           --cv-folds 5           --metric-name accuracy
```

After tuning, reuse the best params JSON saved under `task_a/artifacts/` by loading it into `TRAIN_CONFIG.xgb_params`.

## Saved artifacts

A validation run writes:
- validation metrics JSON
- validation predictions CSV
- XGBoost model JSON
- preprocessor JSON
- joblib bundle metadata
- markdown report in `task_a/reports/`
- submission CSV in both `submissions/` and `task_a/artifacts/` when full-train prediction is enabled
